In [1]:
import numpy as np
import trimesh
import tqdm

In [2]:
def uniform_sampling_from_mesh(vertices, faces, sample_num):
    # -------- TODO -----------
    # 1. compute area of each triangles
    # 2. compute probability of each triangles from areas
    # 3. sample N faces according to the probability
    # 4. for each face, sample 1 point
    # Note that FOR-LOOP is not allowed!
    # -------- TODO -----------
    
    faces_coor = vertices[faces]    # (13712, 3, 3)
    vec1 = faces_coor[:, 1, :] - faces_coor[:, 0, :]    # (13712, 3)
    vec2 = faces_coor[:, 2, :] - faces_coor[:, 0, :]
    area = np.abs(np.sqrt((np.cross(vec1, vec2)**2).sum(axis=1))) / 2  # (13712, )
    prob = area / area.sum()
    sampled_faces_index = np.random.choice(faces.shape[0], sample_num, p=prob)
    sampled_faces_coor = faces_coor[sampled_faces_index, :, :]
    a0 = np.random.uniform(0., 1., (sample_num, 1))
    a1 = np.random.uniform(0., 1., (sample_num, 1))
    uniform_pc = (1 - np.sqrt(a0)) * sampled_faces_coor[:, 0, :] + \
                 np.sqrt(a0) * (1 - a1) * sampled_faces_coor[:, 1, :] + \
                 np.sqrt(a0) * a1 * sampled_faces_coor[:, 2, :] # according to slides
    return area, prob, uniform_pc

In [3]:
def farthest_point_sampling(pc, sample_num):
    # -------- TODO -----------
    # FOR LOOP is allowed here.
    # -------- TODO -----------

    init_point_index = np.random.choice(pc.shape[0], 1)
    results = pc[init_point_index].copy()    # (1, 3)
    results = results[np.newaxis, :, :] # (1, 1, 3)
    pc = pc[:, np.newaxis, :]   # (2000, 1, 3)
    pc = np.delete(pc, init_point_index, 0) # delete initial point from pc to avoid repeats
    for i in tqdm.tqdm(range(sample_num - 1)):
        dis_mat = np.sqrt(np.sum((pc - results)**2, axis=2))  # (1999 - i, i + 1)
        min_dis = np.min(dis_mat, axis=1) # (1999 - i, )
        index = np.argmax(min_dis)
        res = pc[index, :]
        res = res[np.newaxis, :]
        results = np.concatenate([results, res], axis=1)
        pc = np.delete(pc, index, 0)    # delete chosen point from pc to avoid repeats
    return results[0, :, :]

In [4]:
# task 1: uniform sampling 

obj_path = 'spot.obj'
mesh = trimesh.load(obj_path)
print('faces shape:', mesh.faces.shape)
sample_num = 512
area, prob, uniform_pc = uniform_sampling_from_mesh(mesh.vertices, mesh.faces, sample_num)

# Visualization. For you to check your code
np.savetxt('uniform_sampling_vis.txt', uniform_pc)

print('area shape: ',area.shape)
print('prob shape: ',prob.shape)
print('pc shape: ',uniform_pc.shape)
# the result should satisfy: 
#       area.shape = (13712, ) 
#       prob.shape = (13712, ) 
#       uniform_pc.shape = (512, 3)

# For submission
save_dict = {'area': area, 'prob': prob, 'pc': uniform_pc}
np.save('../results/uniform_sampling_results', save_dict)

faces shape: (13712, 3)
area shape:  (13712,)
prob shape:  (13712,)
pc shape:  (512, 3)


In [5]:
# visualization
# import open3d

# visualized_pc = open3d.geometry.PointCloud() 
# visualized_pc.points = open3d.utility.Vector3dVector(uniform_pc)
# open3d.visualization.draw_plotly([visualized_pc], mesh_show_wireframe=False)

In [6]:
# task 2: FPS

init_sample_num = 2000
final_sample_num = 512
_, _, tmp_pc = uniform_sampling_from_mesh(mesh.vertices, mesh.faces, init_sample_num)
fps_pc = farthest_point_sampling(tmp_pc, final_sample_num)

# Visualization. For you to check your code
np.savetxt('fps_vis.txt', fps_pc)

# For submission
np.save('../results/fps_results', fps_pc)

100%|██████████| 511/511 [00:08<00:00, 62.96it/s] 


In [7]:
# visualization
# import open3d

# visualized_pc = open3d.geometry.PointCloud() 
# visualized_pc.points = open3d.utility.Vector3dVector(fps_pc)
# open3d.visualization.draw_plotly([visualized_pc], mesh_show_wireframe=False)

In [8]:
# task 3: metrics

from earthmover.earthmover import earthmover_distance   # EMD may be very slow (1~2mins)
# -----------TODO---------------
# compute chamfer distance and EMD for two point clouds sampled by uniform sampling and FPS.
# sample and compute CD and EMD again. repeat for five times.
# save the mean and var.
# -----------TODO---------------

N = 5
CD = np.zeros(N)
EMD = np.zeros(N)
for i in range(N):
    _, _, tmp_pc_i = uniform_sampling_from_mesh(mesh.vertices, mesh.faces, init_sample_num)
    fps_pc_i = farthest_point_sampling(tmp_pc, final_sample_num)
    _, _, uniform_pc_i = uniform_sampling_from_mesh(mesh.vertices, mesh.faces, final_sample_num)
    fps_pc_i_3d = fps_pc_i[:, np.newaxis, :]
    uniform_pc_i_3d = uniform_pc_i[np.newaxis, :, :]
    dis_mat = np.sqrt(np.sum((fps_pc_i_3d - uniform_pc_i_3d)**2, axis=2))
    CD[i] = np.sum(np.min(dis_mat, axis=1)) / dis_mat.shape[0] + np.sum(np.min(dis_mat, axis=0)) / dis_mat.shape[1]
    fps_pc_i = fps_pc_i.tolist()
    uniform_pc_i = uniform_pc_i.tolist()
    for j in range(final_sample_num):
        fps_pc_i[j] = tuple(fps_pc_i[j])
        uniform_pc_i[j] = tuple(uniform_pc_i[j])
    EMD[i] = earthmover_distance(fps_pc_i, uniform_pc_i)

CD_mean = np.mean(CD)
CD_var = np.var(CD)
EMD_mean = np.mean(EMD)
EMD_var = np.var(EMD)

# For submission
np.save('../results/metrics', {'CD_mean':CD_mean, 'CD_var':CD_var, 'EMD_mean':EMD_mean, 'EMD_var':EMD_var})

100%|██████████| 511/511 [00:06<00:00, 80.75it/s] 


move 0.001953125 dirt from (19.77779546430878, 22.89156157019992, 29.565994110360982) to (22.2024019242466, 18.58280204989158, 23.60494409591175) for a cost of 0.01512609788225802
move 0.001953125 dirt from (51.993099125129724, 9.450562178049786, 38.155139704203215) to (52.286445010466615, 10.711118964920743, 37.73311515303547) for a cost of 0.002658805026310476
move 0.001953125 dirt from (17.704712238892494, 54.68664098827696, 37.25668396339891) to (18.18759565730439, 54.447957646640276, 37.49280252901083) for a cost of 0.0011486934211730555
move 0.001953125 dirt from (45.64650459461051, 33.55614940225064, 27.04836000250871) to (49.66426295995909, 31.628822069849843, 26.860138582267936) for a cost of 0.008711112717107141
move 0.001953125 dirt from (9.532677881139536, 40.369962792591494, 24.089400241024947) to (8.712913819132243, 39.5189343503636, 24.288451745573713) for a cost of 0.0023403980166887767
move 0.001953125 dirt from (37.09775655264766, 23.413137306640927, 42.44631329811004

100%|██████████| 511/511 [00:05<00:00, 88.59it/s] 


move 0.001953125 dirt from (21.56816523249777, 34.86492277072783, 37.72364594306592) to (21.09912282733264, 34.384935215878315, 37.610195169998136) for a cost of 0.001329359359417033
move 0.001953125 dirt from (52.06932695666323, 9.403199728472703, 25.504160784558845) to (52.13544542679328, 9.449205193862872, 26.09804032784239) for a cost of 0.0011705413421355922
move 0.001953125 dirt from (23.441576865934664, 9.906538871539984, 22.57718978897867) to (25.806878418149317, 10.92770186297556, 20.847638785120413) for a cost of 0.006060598161760131
move 0.001953125 dirt from (47.39459838129597, 32.78970208869251, 36.048980657222735) to (47.901971349233214, 32.58932592294313, 36.03588726187597) for a cost of 0.001065750308718537
move 0.001953125 dirt from (17.84959468396759, 48.98399299111764, 18.87882130002797) to (21.87427080927535, 45.80110016654358, 21.97722098912794) for a cost of 0.011707172875105596
move 0.001953125 dirt from (33.35794533890188, 16.799239849925797, 41.15190407104626) 

100%|██████████| 511/511 [00:05<00:00, 88.11it/s] 


move 0.001953125 dirt from (32.04145329129406, 20.556309380937442, 42.668799042857714) to (34.424892496217026, 15.415385505329539, 38.50680717384908) for a cost of 0.01373201852918736
move 0.001953125 dirt from (17.348065210416472, 54.94992758513631, 26.470205352866255) to (17.009231320298856, 54.15032712257739, 27.66647481560316) for a cost of 0.0028872115695740045
move 0.001953125 dirt from (51.264340536258835, 9.9469877325665, 23.514747521758274) to (50.01824228481952, 10.860200278117228, 22.25693627910081) for a cost of 0.0038909892955402733
move 0.001953125 dirt from (9.20522647209658, 32.83884017974453, 38.58019247608433) to (8.575540264225616, 35.10087199724938, 39.9472835122988) for a cost of 0.005306691507035593
move 0.001953125 dirt from (24.444247194074784, 10.52685953402052, 21.440017702387685) to (24.286050986756237, 11.258590385069152, 21.21693122535779) for a cost of 0.0015257190130225052
move 0.001953125 dirt from (45.64650459461051, 33.55614940225064, 27.04836000250871

100%|██████████| 511/511 [00:05<00:00, 88.48it/s] 


move 0.001953125 dirt from (16.85353034669666, 32.74721677524866, 24.941961072956623) to (16.787995743036607, 33.29953221542254, 24.26971169636084) for a cost of 0.0017041128716323833
move 0.001953125 dirt from (51.516646104329745, 8.541205595157379, 37.91094167952957) to (44.62154115915787, 11.953716685619007, 39.85600324280313) for a cost of 0.015498876793358384
move 0.001953125 dirt from (23.353961374939686, 10.095686378314719, 40.736312821124656) to (22.755813369228616, 9.99888969361928, 39.67434520234126) for a cost of 0.0023880304821700246
move 0.001953125 dirt from (45.64650459461051, 33.55614940225064, 27.04836000250871) to (45.96323974798044, 34.07637253152297, 28.44534237656884) for a cost of 0.002976522139657295
move 0.001953125 dirt from (19.17943249419224, 48.323961596654286, 44.714195260827175) to (19.488262139429338, 49.11302828127123, 43.895000251060694) for a cost of 0.002301940270876061
move 0.001953125 dirt from (29.646154586847626, 15.757569724682785, 20.89677286250

100%|██████████| 511/511 [00:05<00:00, 88.67it/s] 


move 0.001953125 dirt from (36.840760928048276, 15.643856521766663, 39.61483969289009) to (33.65987538926311, 15.480078117709226, 38.72886533651501) for a cost of 0.0064570816838630495
move 0.001953125 dirt from (17.348065210416472, 54.94992758513631, 26.470205352866255) to (17.444919626897477, 54.999367456587855, 26.399126416488446) for a cost of 0.00025373562534467605
move 0.001953125 dirt from (11.637306477234546, 31.339942749997665, 37.97841711884308) to (11.42524384850778, 31.019450709697733, 36.40959542935052) for a cost of 0.0031546972703381626
move 0.001953125 dirt from (32.81316159467612, 35.56754455594976, 25.24337360772558) to (32.14177695285177, 35.05900026461673, 24.644343824436316) for a cost of 0.0020186388437878214
move 0.001953125 dirt from (53.845081491872335, 26.332544245113024, 28.59391159709842) to (53.52918343308785, 25.346595551412204, 28.06244316319031) for a cost of 0.002272975269130136
move 0.001953125 dirt from (23.504018817649566, 14.889691350830626, 21.1722

In [9]:
CD_mean

2.733733148930579

In [10]:
CD_var

0.002022915058730834

In [11]:
EMD_mean

2.350284581275141

In [12]:
EMD_var

0.03828230324148696